# 10 — Core directionality and CYP24A1 audit

## Purpose

This revision notebook audits the directionality and membership of the canonical vitamin D transcriptional core in response to reviewer comments.

The analysis focuses on three related questions:

1. Whether the genes included in the canonical core are regulated in the expected direction across the five analyzed cell lines.
2. Why `CYP24A1`, a canonical vitamin D target gene, does not appear among the top genes shown in Figure 2A.
3. Why `CYP24A1` is not included in the shared/core gene set under the manuscript-defined recurrence criteria.

## Reviewer motivation

Reviewer 1 raised specific concerns about the interpretation of gene-level modulation, the absence of `CYP24A1` from Figure 2A, and whether the shared vitamin D-responsive genes are regulated in the same direction across all five cell lines.

This notebook is intended to generate compact revision tables that can support the response letter, manuscript text clarification, and possible figure/legend revision.

## Output policy

This notebook does not write persistent output tables by default.

The analysis generates in-memory audit tables and printed summaries to support reviewer response drafting and manuscript revision. Tables should only be exported later if they are explicitly promoted to supplementary material or required as consumable inputs for another revision notebook.

Potential promotable tables include:

- `core_directionality_summary`
- `cyp24a1_ranking_summary`
- `cyp24a1_core_membership_check`

## Interpretation boundary

The goal is not to redefine the vitamin D transcriptional core, but to audit the existing canonical definition and explain its behavior transparently.


In [ ]:
# =============================================================================
# Notebook setup
# =============================================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# Detect project root
# -----------------------------------------------------------------------------
# The notebook may be executed from notebooks/revision/ or from the repository
# root. We therefore walk upward until both src/ and data/ are found.

current_path = Path.cwd().resolve()
project_root = None

for candidate in [current_path, *current_path.parents]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError(
        "Could not detect project root. Expected to find both 'src/' and 'data/' "
        "in the current directory or one of its parent directories."
    )


# -----------------------------------------------------------------------------
# Make local project utilities importable
# -----------------------------------------------------------------------------

src_dir = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))


# -----------------------------------------------------------------------------
# Import project utilities
# -----------------------------------------------------------------------------
# coregenes contains the canonical consensus-core construction functions.
# idsymbols is used later for stable gene ID to gene symbol mapping.
# config stores canonical manuscript parameters when available.

from vitd_utils import config
from vitd_utils import coregenes
from vitd_utils import idsymbols


# -----------------------------------------------------------------------------
# Define input paths
# -----------------------------------------------------------------------------

data_exports_dir = project_root / "data" / "exports"
raw_data_dir = project_root / "data" / "raw_data"

expression_path = data_exports_dir / "expression_matrix_clean.parquet"
metadata_path = data_exports_dir / "signature_metadata_clean.csv"
geneinfo_path = raw_data_dir / "geneinfo_beta.txt"


# -----------------------------------------------------------------------------
# Define revision output paths
# -----------------------------------------------------------------------------

revision_tables_dir = project_root / "results" / "revision" / "tables"

core_directionality_output_path = revision_tables_dir / "core_directionality_summary.csv"
cyp24a1_ranking_output_path = revision_tables_dir / "cyp24a1_ranking_summary.csv"
cyp24a1_membership_output_path = revision_tables_dir / "cyp24a1_core_membership_check.csv"


# -----------------------------------------------------------------------------
# Create output directory
# -----------------------------------------------------------------------------
# Creating only the output folder is safe here because this notebook is expected
# to write revision tables under a dedicated namespace.

revision_tables_dir.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Report resolved paths
# -----------------------------------------------------------------------------

print("Notebook setup complete.")
print(f"Project root: {project_root}")
print(f"Expression matrix path: {expression_path}")
print(f"Metadata path: {metadata_path}")
print(f"Gene info path: {geneinfo_path}")
print(f"Revision tables directory: {revision_tables_dir}")

In [ ]:
# =============================================================================
# Load and validate input data
# =============================================================================
# This cell loads the cleaned expression matrix, cleaned signature metadata, and
# LINCS gene annotation table. It performs strict integrity checks before any
# downstream analysis:
#
#   1. All required files must exist.
#   2. Metadata must contain the required signature and cell-line columns.
#   3. Signature identifiers must be unique.
#   4. Expression columns must map exactly to metadata signature identifiers.
#   5. Metadata is reordered to match the expression matrix column order.
#
# The expected expression matrix orientation is:
#   rows    = genes
#   columns = LINCS signatures
# =============================================================================


# -----------------------------------------------------------------------------
# Check required input files
# -----------------------------------------------------------------------------

required_input_paths = {
    "expression_matrix_clean": expression_path,
    "signature_metadata_clean": metadata_path,
    "geneinfo_beta": geneinfo_path,
}

missing_inputs = [
    name for name, path in required_input_paths.items()
    if not path.exists()
]

if missing_inputs:
    missing_report = "\n".join(
        f"- {name}: {required_input_paths[name]}"
        for name in missing_inputs
    )
    raise FileNotFoundError(
        "Missing required input file(s):\n"
        f"{missing_report}"
    )


# -----------------------------------------------------------------------------
# Load data
# -----------------------------------------------------------------------------

exp = pd.read_parquet(expression_path)
meta = pd.read_csv(metadata_path)
geneinfo = pd.read_csv(geneinfo_path, sep="\t")


# -----------------------------------------------------------------------------
# Validate metadata schema
# -----------------------------------------------------------------------------

required_metadata_columns = {"sig_id", "cell_id"}
missing_metadata_columns = required_metadata_columns - set(meta.columns)

if missing_metadata_columns:
    raise ValueError(
        "Metadata is missing required column(s): "
        f"{sorted(missing_metadata_columns)}"
    )

if meta["sig_id"].isna().any():
    raise ValueError("Metadata contains missing sig_id values.")

if meta["cell_id"].isna().any():
    raise ValueError("Metadata contains missing cell_id values.")

if meta["sig_id"].duplicated().any():
    duplicated_sig_ids = meta.loc[meta["sig_id"].duplicated(), "sig_id"].tolist()
    raise ValueError(
        "Metadata contains duplicated sig_id values. "
        f"First duplicated examples: {duplicated_sig_ids[:10]}"
    )


# -----------------------------------------------------------------------------
# Validate expression matrix structure
# -----------------------------------------------------------------------------

if exp.empty:
    raise ValueError("Expression matrix is empty.")

if exp.index.has_duplicates:
    duplicated_gene_ids = exp.index[exp.index.duplicated()].tolist()
    raise ValueError(
        "Expression matrix contains duplicated gene identifiers. "
        f"First duplicated examples: {duplicated_gene_ids[:10]}"
    )

if exp.columns.has_duplicates:
    duplicated_columns = exp.columns[exp.columns.duplicated()].tolist()
    raise ValueError(
        "Expression matrix contains duplicated signature columns. "
        f"First duplicated examples: {duplicated_columns[:10]}"
    )

missing_expression_columns_in_metadata = sorted(set(exp.columns) - set(meta["sig_id"]))
extra_metadata_signatures = sorted(set(meta["sig_id"]) - set(exp.columns))

if missing_expression_columns_in_metadata:
    raise ValueError(
        "Some expression matrix columns are missing from metadata. "
        f"Number missing: {len(missing_expression_columns_in_metadata)}. "
        f"First examples: {missing_expression_columns_in_metadata[:10]}"
    )


# -----------------------------------------------------------------------------
# Align metadata to expression matrix column order
# -----------------------------------------------------------------------------

meta = (
    meta
    .set_index("sig_id")
    .loc[list(exp.columns)]
    .reset_index()
)

if list(meta["sig_id"]) != list(exp.columns):
    raise AssertionError(
        "Metadata alignment failed: metadata sig_id order does not match "
        "expression matrix column order."
    )


# -----------------------------------------------------------------------------
# Report loaded data
# -----------------------------------------------------------------------------

print("Input data loaded and validated.")
print(f"Expression matrix shape: {exp.shape[0]} genes x {exp.shape[1]} signatures")
print(f"Metadata shape: {meta.shape[0]} signatures x {meta.shape[1]} columns")
print(f"Gene annotation shape: {geneinfo.shape[0]} rows x {geneinfo.shape[1]} columns")
print("")
print(f"Metadata signatures not used by expression matrix: {len(extra_metadata_signatures)}")
print("")
print("Signatures per cell line:")
print(meta["cell_id"].value_counts().sort_index().to_string())

In [ ]:
# =============================================================================
# Reconstruct the canonical vitamin D transcriptional core
# =============================================================================
# This cell reconstructs the manuscript-defined consensus core from the cleaned
# expression matrix and aligned metadata.
#
# The canonical workflow is:
#   1. Average LINCS Level 5 z-scores within each cell line.
#   2. Select the strongest positive and negative genes per cell line.
#   3. Count recurrence across cell lines separately for UP and DOWN directions.
#   4. Retain the canonical target sizes used in the manuscript:
#        - 42 UP genes
#        - 35 DOWN genes
#
# This cell does not redefine the core. It reproduces the existing canonical
# definition so that directionality and CYP24A1 membership can be audited.
# =============================================================================


# -----------------------------------------------------------------------------
# Canonical core parameters
# -----------------------------------------------------------------------------
# These values correspond to the manuscript-facing directed-results workflow.

canonical_top_n = 50
canonical_min_votes = 2
canonical_target_up = 42
canonical_target_down = 35


# -----------------------------------------------------------------------------
# Build gene-by-cell-line mean effect matrix
# -----------------------------------------------------------------------------
# Input expression matrix orientation:
#   exp: genes x signatures
#
# After transposition:
#   exp.T: signatures x genes
#
# We append cell_id and average all signatures within each cell line.

effects_by_cell = (
    exp.T
    .assign(cell_id=meta["cell_id"].to_numpy())
    .groupby("cell_id")
    .mean()
    .T
)


# -----------------------------------------------------------------------------
# Reconstruct canonical consensus core
# -----------------------------------------------------------------------------

canonical_core = coregenes.build_consensus_core(
    effects_by_cell,
    top_n=canonical_top_n,
    min_votes=canonical_min_votes,
    target_up=canonical_target_up,
    target_dn=canonical_target_down,
)


# -----------------------------------------------------------------------------
# Extract UP and DOWN gene identifiers
# -----------------------------------------------------------------------------
# Current vitd_utils.coregenes.build_consensus_core returns:
#   - core_up
#   - core_dn
#   - votes_up
#   - votes_dn
#   - extremes_by_col
#   - summary
#
# We keep the extraction explicit to avoid silently accepting an unexpected
# object schema.

expected_core_keys = {"core_up", "core_dn", "votes_up", "votes_dn", "extremes_by_col", "summary"}
observed_core_keys = set(canonical_core.keys())

missing_core_keys = expected_core_keys - observed_core_keys
if missing_core_keys:
    raise KeyError(
        "Canonical core object is missing expected key(s): "
        f"{sorted(missing_core_keys)}. "
        f"Available keys: {sorted(observed_core_keys)}"
    )

core_up_ids = list(canonical_core["core_up"])
core_down_ids = list(canonical_core["core_dn"])


# -----------------------------------------------------------------------------
# Validate expected canonical core sizes
# -----------------------------------------------------------------------------

if len(core_up_ids) != canonical_target_up:
    raise AssertionError(
        f"Unexpected UP core size: {len(core_up_ids)}. "
        f"Expected: {canonical_target_up}."
    )

if len(core_down_ids) != canonical_target_down:
    raise AssertionError(
        f"Unexpected DOWN core size: {len(core_down_ids)}. "
        f"Expected: {canonical_target_down}."
    )

overlapping_core_ids = sorted(set(core_up_ids).intersection(core_down_ids))

if overlapping_core_ids:
    raise AssertionError(
        "Some genes are present in both UP and DOWN core sets. "
        f"Examples: {overlapping_core_ids[:10]}"
    )


# -----------------------------------------------------------------------------
# Report results
# -----------------------------------------------------------------------------

print("Canonical core reconstructed successfully.")
print("")
print("Canonical parameters:")
print(f"  top_n per direction per cell line: {canonical_top_n}")
print(f"  minimum recurrence votes: {canonical_min_votes}")
print(f"  target UP genes: {canonical_target_up}")
print(f"  target DOWN genes: {canonical_target_down}")
print("")
print(f"Effects-by-cell matrix shape: {effects_by_cell.shape[0]} genes x {effects_by_cell.shape[1]} cell lines")
print(f"Cell lines: {', '.join(effects_by_cell.columns.astype(str))}")
print("")
print(f"Core UP genes: {len(core_up_ids)}")
print(f"Core DOWN genes: {len(core_down_ids)}")
print(f"Total core genes: {len(core_up_ids) + len(core_down_ids)}")
print("")
print("Canonical core object keys:")
print(sorted(observed_core_keys))
print("")
print("First 10 UP gene IDs:")
print(core_up_ids[:10])
print("")
print("First 10 DOWN gene IDs:")
print(core_down_ids[:10])

In [ ]:
# =============================================================================
# Build gene ID to gene symbol annotations
# =============================================================================
# This cell constructs a stable mapping between LINCS gene identifiers and gene
# symbols using the loaded gene annotation table.
#
# The expression matrix and the canonical core use LINCS gene IDs as identifiers.
# Reviewer-facing interpretation, however, requires gene symbols. This mapping
# will be used downstream to annotate:
#
#   - canonical core genes
#   - CYP24A1 ranking and membership checks
#   - directionality summaries
#
# No filtering is applied to the core in this cell.
# =============================================================================


# -----------------------------------------------------------------------------
# Detect required gene annotation columns
# -----------------------------------------------------------------------------
# LINCS geneinfo files usually use:
#   - pr_gene_id
#   - pr_gene_symbol
#
# The detection logic keeps the notebook robust to minor naming differences while
# still failing explicitly if no suitable columns are found.

candidate_gene_id_columns = [
    "pr_gene_id",
    "gene_id",
    "id",
]

candidate_gene_symbol_columns = [
    "pr_gene_symbol",
    "gene_symbol",
    "symbol",
]

candidate_landmark_columns = [
    "pr_is_lm",
    "is_landmark",
    "landmark",
]


def find_first_existing_column(frame, candidate_columns, column_role):
    """Return the first matching column name from a list of candidates."""
    for column in candidate_columns:
        if column in frame.columns:
            return column
    
    raise ValueError(
        f"Could not identify a {column_role} column in geneinfo. "
        f"Candidate columns checked: {candidate_columns}. "
        f"Available columns: {list(frame.columns)}"
    )


gene_id_column = find_first_existing_column(
    geneinfo,
    candidate_gene_id_columns,
    "gene identifier",
)

gene_symbol_column = find_first_existing_column(
    geneinfo,
    candidate_gene_symbol_columns,
    "gene symbol",
)

landmark_column = None
for column in candidate_landmark_columns:
    if column in geneinfo.columns:
        landmark_column = column
        break


# -----------------------------------------------------------------------------
# Construct normalized annotation table
# -----------------------------------------------------------------------------
# Gene IDs are stored as strings for reliable joins across pandas objects that
# may otherwise mix integer and string representations.

annotation_columns = [gene_id_column, gene_symbol_column]

if landmark_column is not None:
    annotation_columns.append(landmark_column)

gene_annotation = geneinfo.loc[:, annotation_columns].copy()

rename_map = {
    gene_id_column: "gene_id",
    gene_symbol_column: "gene_symbol",
}

if landmark_column is not None:
    rename_map[landmark_column] = "is_landmark"

gene_annotation = gene_annotation.rename(columns=rename_map)

gene_annotation["gene_id"] = gene_annotation["gene_id"].astype(str)
gene_annotation["gene_symbol"] = gene_annotation["gene_symbol"].astype(str)

if "is_landmark" not in gene_annotation.columns:
    gene_annotation["is_landmark"] = np.nan


# -----------------------------------------------------------------------------
# Validate annotation uniqueness and expression coverage
# -----------------------------------------------------------------------------

if gene_annotation["gene_id"].duplicated().any():
    duplicated_gene_annotation_ids = (
        gene_annotation
        .loc[gene_annotation["gene_id"].duplicated(), "gene_id"]
        .tolist()
    )
    raise ValueError(
        "Gene annotation table contains duplicated gene IDs. "
        f"First duplicated examples: {duplicated_gene_annotation_ids[:10]}"
    )

expression_gene_ids = pd.Index(exp.index.astype(str))
annotated_gene_ids = set(gene_annotation["gene_id"])

missing_expression_gene_annotations = sorted(
    set(expression_gene_ids) - annotated_gene_ids
)

if missing_expression_gene_annotations:
    raise ValueError(
        "Some expression matrix gene IDs are missing from gene annotation. "
        f"Number missing: {len(missing_expression_gene_annotations)}. "
        f"First examples: {missing_expression_gene_annotations[:10]}"
    )


# -----------------------------------------------------------------------------
# Build mapping dictionaries
# -----------------------------------------------------------------------------

gene_id_to_symbol = dict(
    zip(gene_annotation["gene_id"], gene_annotation["gene_symbol"])
)

gene_id_to_landmark = dict(
    zip(gene_annotation["gene_id"], gene_annotation["is_landmark"])
)

symbol_to_gene_ids = (
    gene_annotation
    .groupby("gene_symbol")["gene_id"]
    .apply(list)
    .to_dict()
)


# -----------------------------------------------------------------------------
# Annotate canonical core genes
# -----------------------------------------------------------------------------

core_up_ids_str = [str(gene_id) for gene_id in core_up_ids]
core_down_ids_str = [str(gene_id) for gene_id in core_down_ids]

core_gene_table = pd.DataFrame(
    {
        "gene_id": core_up_ids_str + core_down_ids_str,
        "core_direction": (
            ["UP"] * len(core_up_ids_str)
            + ["DOWN"] * len(core_down_ids_str)
        ),
    }
)

core_gene_table["gene_symbol"] = core_gene_table["gene_id"].map(gene_id_to_symbol)
core_gene_table["is_landmark"] = core_gene_table["gene_id"].map(gene_id_to_landmark)

if core_gene_table["gene_symbol"].isna().any():
    missing_core_symbols = (
        core_gene_table
        .loc[core_gene_table["gene_symbol"].isna(), "gene_id"]
        .tolist()
    )
    raise ValueError(
        "Some core gene IDs could not be mapped to gene symbols. "
        f"Examples: {missing_core_symbols[:10]}"
    )


# -----------------------------------------------------------------------------
# Report annotation status
# -----------------------------------------------------------------------------

duplicated_symbols = (
    gene_annotation["gene_symbol"]
    .value_counts()
    .loc[lambda counts: counts > 1]
)

print("Gene annotation mapping completed.")
print("")
print(f"Gene ID column used: {gene_id_column}")
print(f"Gene symbol column used: {gene_symbol_column}")
print(f"Landmark column used: {landmark_column}")
print("")
print(f"Annotated genes: {gene_annotation.shape[0]}")
print(f"Expression genes covered by annotation: {len(expression_gene_ids)}")
print(f"Duplicated gene symbols in annotation: {duplicated_symbols.shape[0]}")
print("")
print("Canonical core annotation summary:")
print(core_gene_table["core_direction"].value_counts().sort_index().to_string())
print("")
print("First 10 annotated core genes:")
print(core_gene_table.head(10).to_string(index=False))

In [ ]:
# =============================================================================
# Audit directionality of canonical core genes across cell lines
# =============================================================================
# This cell evaluates whether each canonical core gene follows the expected
# direction of regulation across the five analyzed cell lines.
#
# Expected direction is defined by canonical core membership:
#   - UP core genes are expected to have positive mean effects.
#   - DOWN core genes are expected to have negative mean effects.
#
# For each gene, the cell computes:
#   - mean effect by cell line
#   - number of cell lines matching the expected direction
#   - number of cell lines showing the opposite direction
#   - number of neutral cell lines, if any
#   - a sign-based directionality class
#
# No arbitrary effect-size threshold is applied here. Classification is based
# only on the sign of the mean cell-line effect.
#
# No files are saved in this cell.
# =============================================================================


# -----------------------------------------------------------------------------
# Prepare effects matrix with string gene IDs
# -----------------------------------------------------------------------------

effects_by_cell_str = effects_by_cell.copy()
effects_by_cell_str.index = effects_by_cell_str.index.astype(str)

missing_core_ids_in_effects = sorted(
    set(core_gene_table["gene_id"]) - set(effects_by_cell_str.index)
)

if missing_core_ids_in_effects:
    raise ValueError(
        "Some canonical core gene IDs are missing from effects_by_cell. "
        f"Number missing: {len(missing_core_ids_in_effects)}. "
        f"First examples: {missing_core_ids_in_effects[:10]}"
    )


# -----------------------------------------------------------------------------
# Build core directionality table
# -----------------------------------------------------------------------------

cell_line_columns = list(effects_by_cell_str.columns)

core_directionality_summary = core_gene_table.copy()

for cell_line in cell_line_columns:
    core_directionality_summary[f"mean_effect_{cell_line}"] = (
        core_directionality_summary["gene_id"]
        .map(effects_by_cell_str[cell_line])
    )


# -----------------------------------------------------------------------------
# Count expected, opposite, and neutral directions
# -----------------------------------------------------------------------------

def summarize_directionality(row):
    """Summarize sign concordance for one canonical core gene."""
    expected_direction = row["core_direction"]
    effects = row[[f"mean_effect_{cell}" for cell in cell_line_columns]].astype(float)
    
    if expected_direction == "UP":
        expected_count = int((effects > 0).sum())
        opposite_count = int((effects < 0).sum())
    elif expected_direction == "DOWN":
        expected_count = int((effects < 0).sum())
        opposite_count = int((effects > 0).sum())
    else:
        raise ValueError(f"Unexpected core direction: {expected_direction}")
    
    neutral_count = int((effects == 0).sum())
    
    if expected_count == len(cell_line_columns):
        directionality_class = "concordant_all_5"
    elif expected_count == 4:
        directionality_class = "concordant_4_of_5"
    elif expected_count >= canonical_min_votes:
        directionality_class = "partially_concordant"
    elif opposite_count > expected_count:
        directionality_class = "discordant_or_opposite_dominant"
    else:
        directionality_class = "weak_or_mixed"
    
    return pd.Series(
        {
            "expected_direction_count": expected_count,
            "opposite_direction_count": opposite_count,
            "neutral_direction_count": neutral_count,
            "directionality_class": directionality_class,
            "mean_effect_across_cells": float(effects.mean()),
            "mean_abs_effect_across_cells": float(effects.abs().mean()),
            "min_abs_effect_across_cells": float(effects.abs().min()),
            "max_abs_effect_across_cells": float(effects.abs().max()),
        }
    )


directionality_metrics = core_directionality_summary.apply(
    summarize_directionality,
    axis=1,
)

core_directionality_summary = pd.concat(
    [core_directionality_summary, directionality_metrics],
    axis=1,
)


# -----------------------------------------------------------------------------
# Order columns for readability
# -----------------------------------------------------------------------------

ordered_columns = (
    ["gene_id", "gene_symbol", "core_direction", "is_landmark"]
    + [f"mean_effect_{cell}" for cell in cell_line_columns]
    + [
        "expected_direction_count",
        "opposite_direction_count",
        "neutral_direction_count",
        "directionality_class",
        "mean_effect_across_cells",
        "mean_abs_effect_across_cells",
        "min_abs_effect_across_cells",
        "max_abs_effect_across_cells",
    ]
)

core_directionality_summary = core_directionality_summary.loc[:, ordered_columns]


# -----------------------------------------------------------------------------
# Report directionality audit
# -----------------------------------------------------------------------------

print("Core directionality audit completed.")
print("")
print(f"Core genes evaluated: {core_directionality_summary.shape[0]}")
print(f"Cell lines evaluated: {len(cell_line_columns)}")
print("")
print("Directionality classes:")
print(
    core_directionality_summary["directionality_class"]
    .value_counts()
    .sort_index()
    .to_string()
)
print("")
print("Directionality classes by canonical core direction:")
print(
    pd.crosstab(
        core_directionality_summary["core_direction"],
        core_directionality_summary["directionality_class"],
    )
    .to_string()
)
print("")
print("First 15 rows:")
print(core_directionality_summary.head(15).to_string(index=False))

In [ ]:
# =============================================================================
# Audit CYP24A1 ranking under the Figure 2A gene-level criterion
# =============================================================================
# Figure 2A ranks genes by global mean absolute LINCS Level 5 z-score across all
# curated vitamin D perturbation signatures. This criterion captures response
# magnitude regardless of direction and does not prioritize canonical vitamin D
# target status.
#
# This cell evaluates CYP24A1 under that same criterion:
#   - whether CYP24A1 is present in the LINCS expression matrix
#   - its global mean absolute z-score
#   - its global mean signed z-score
#   - its rank by mean absolute z-score
#   - its rank by positive signed mean effect
#   - its cell-line-specific mean effects
#
# No files are saved in this cell.
# =============================================================================


# -----------------------------------------------------------------------------
# Locate CYP24A1 in the annotation table and expression matrix
# -----------------------------------------------------------------------------

target_gene_symbol = "CYP24A1"

target_gene_ids = symbol_to_gene_ids.get(target_gene_symbol, [])
target_gene_ids = [str(gene_id) for gene_id in target_gene_ids]

exp_for_ranking = exp.copy()
exp_for_ranking.index = exp_for_ranking.index.astype(str)

target_gene_ids_in_expression = [
    gene_id for gene_id in target_gene_ids
    if gene_id in exp_for_ranking.index
]

if not target_gene_ids:
    raise ValueError(
        f"{target_gene_symbol} was not found in the gene annotation table."
    )

if not target_gene_ids_in_expression:
    raise ValueError(
        f"{target_gene_symbol} was found in gene annotation but not in the "
        "expression matrix. "
        f"Annotated gene IDs: {target_gene_ids}"
    )

if len(target_gene_ids_in_expression) > 1:
    raise ValueError(
        f"Multiple expression gene IDs map to {target_gene_symbol}. "
        "Manual disambiguation is required before continuing. "
        f"Gene IDs: {target_gene_ids_in_expression}"
    )

target_gene_id = target_gene_ids_in_expression[0]


# -----------------------------------------------------------------------------
# Compute global gene-level ranking metrics
# -----------------------------------------------------------------------------

mean_abs_z = exp_for_ranking.abs().mean(axis=1)
mean_signed_z = exp_for_ranking.mean(axis=1)

ranking_table = pd.DataFrame(
    {
        "gene_id": mean_abs_z.index,
        "gene_symbol": mean_abs_z.index.map(gene_id_to_symbol),
        "mean_abs_z": mean_abs_z.to_numpy(),
        "mean_signed_z": mean_signed_z.to_numpy(),
    }
)

ranking_table["rank_mean_abs_z"] = (
    ranking_table["mean_abs_z"]
    .rank(method="min", ascending=False)
    .astype(int)
)

ranking_table["rank_positive_mean_signed_z"] = (
    ranking_table["mean_signed_z"]
    .rank(method="min", ascending=False)
    .astype(int)
)

ranking_table["rank_negative_mean_signed_z"] = (
    ranking_table["mean_signed_z"]
    .rank(method="min", ascending=True)
    .astype(int)
)


# -----------------------------------------------------------------------------
# Extract CYP24A1 ranking row
# -----------------------------------------------------------------------------

cyp24a1_ranking_summary = (
    ranking_table
    .loc[ranking_table["gene_id"] == target_gene_id]
    .copy()
)

if cyp24a1_ranking_summary.empty:
    raise AssertionError(
        "CYP24A1 ranking row could not be extracted after successful ID mapping."
    )


# -----------------------------------------------------------------------------
# Add cell-line-specific mean effects
# -----------------------------------------------------------------------------

for cell_line in cell_line_columns:
    cyp24a1_ranking_summary[f"mean_effect_{cell_line}"] = (
        effects_by_cell_str.loc[target_gene_id, cell_line]
    )

cyp24a1_ranking_summary["n_signatures"] = exp_for_ranking.shape[1]
cyp24a1_ranking_summary["n_genes_ranked"] = exp_for_ranking.shape[0]
cyp24a1_ranking_summary["in_core_up"] = target_gene_id in set(core_up_ids_str)
cyp24a1_ranking_summary["in_core_down"] = target_gene_id in set(core_down_ids_str)
cyp24a1_ranking_summary["in_canonical_core"] = (
    cyp24a1_ranking_summary["in_core_up"]
    | cyp24a1_ranking_summary["in_core_down"]
)

ordered_cyp24a1_columns = (
    [
        "gene_id",
        "gene_symbol",
        "n_signatures",
        "n_genes_ranked",
        "mean_abs_z",
        "rank_mean_abs_z",
        "mean_signed_z",
        "rank_positive_mean_signed_z",
        "rank_negative_mean_signed_z",
    ]
    + [f"mean_effect_{cell}" for cell in cell_line_columns]
    + [
        "in_core_up",
        "in_core_down",
        "in_canonical_core",
    ]
)

cyp24a1_ranking_summary = cyp24a1_ranking_summary.loc[
    :,
    ordered_cyp24a1_columns,
]


# -----------------------------------------------------------------------------
# Report CYP24A1 ranking audit
# -----------------------------------------------------------------------------

print("CYP24A1 ranking audit completed.")
print("")
print(f"Target gene symbol: {target_gene_symbol}")
print(f"Target gene ID: {target_gene_id}")
print(f"Annotated gene IDs for {target_gene_symbol}: {target_gene_ids}")
print("")
print("CYP24A1 ranking summary:")
print(cyp24a1_ranking_summary.to_string(index=False))
print("")
print("Top 20 genes by Figure 2A criterion: mean_abs_z")
print(
    ranking_table
    .sort_values("rank_mean_abs_z")
    .head(20)
    .loc[:, ["gene_id", "gene_symbol", "mean_abs_z", "mean_signed_z", "rank_mean_abs_z"]]
    .to_string(index=False)
)

In [ ]:
# =============================================================================
# Audit CYP24A1 membership under the canonical core-selection procedure
# =============================================================================
# The canonical core is not selected from global Figure 2A ranks. It is selected
# by recurrent cell-line-level extremes:
#
#   1. For each cell line, genes are ranked by mean signed effect.
#   2. The top 50 positive and top 50 negative genes are selected per cell line.
#   3. Genes must recur in at least 2 cell lines in the same direction.
#   4. The final UP and DOWN core sets are selected from those recurrent genes.
#
# This cell evaluates CYP24A1 under that exact recurrence logic and identifies
# the step at which it fails or passes.
#
# No files are saved in this cell.
# =============================================================================


# -----------------------------------------------------------------------------
# Validate target gene availability
# -----------------------------------------------------------------------------

if target_gene_id not in effects_by_cell_str.index:
    raise ValueError(
        f"{target_gene_symbol} gene ID {target_gene_id} is not present in "
        "effects_by_cell_str."
    )


# -----------------------------------------------------------------------------
# Compute per-cell-line ranks and top-N membership for CYP24A1
# -----------------------------------------------------------------------------

cyp24a1_membership_records = []

for cell_line in cell_line_columns:
    cell_effects = effects_by_cell_str[cell_line].astype(float).dropna()
    
    if target_gene_id not in cell_effects.index:
        raise ValueError(
            f"{target_gene_symbol} gene ID {target_gene_id} is missing from "
            f"cell-line effects for {cell_line}."
        )
    
    positive_ranked_genes = cell_effects.sort_values(ascending=False)
    negative_ranked_genes = cell_effects.sort_values(ascending=True)
    
    positive_rank = int(positive_ranked_genes.index.get_loc(target_gene_id) + 1)
    negative_rank = int(negative_ranked_genes.index.get_loc(target_gene_id) + 1)
    
    top_up_gene_ids = set(positive_ranked_genes.head(canonical_top_n).index.astype(str))
    top_down_gene_ids = set(negative_ranked_genes.head(canonical_top_n).index.astype(str))
    
    mean_effect = float(cell_effects.loc[target_gene_id])
    
    if mean_effect > 0:
        signed_direction = "positive"
    elif mean_effect < 0:
        signed_direction = "negative"
    else:
        signed_direction = "zero"
    
    cyp24a1_membership_records.append(
        {
            "gene_id": target_gene_id,
            "gene_symbol": target_gene_symbol,
            "cell_id": cell_line,
            "mean_effect": mean_effect,
            "signed_direction": signed_direction,
            "rank_positive_effect": positive_rank,
            "rank_negative_effect": negative_rank,
            f"in_top{canonical_top_n}_up": target_gene_id in top_up_gene_ids,
            f"in_top{canonical_top_n}_down": target_gene_id in top_down_gene_ids,
        }
    )

cyp24a1_membership_by_cell = pd.DataFrame(cyp24a1_membership_records)


# -----------------------------------------------------------------------------
# Summarize recurrence votes
# -----------------------------------------------------------------------------

top_up_column = f"in_top{canonical_top_n}_up"
top_down_column = f"in_top{canonical_top_n}_down"

cyp24a1_up_votes = int(cyp24a1_membership_by_cell[top_up_column].sum())
cyp24a1_down_votes = int(cyp24a1_membership_by_cell[top_down_column].sum())

passes_up_recurrence = cyp24a1_up_votes >= canonical_min_votes
passes_down_recurrence = cyp24a1_down_votes >= canonical_min_votes

in_core_up = target_gene_id in set(core_up_ids_str)
in_core_down = target_gene_id in set(core_down_ids_str)
in_canonical_core = in_core_up or in_core_down


# -----------------------------------------------------------------------------
# Determine core-selection outcome
# -----------------------------------------------------------------------------

if in_core_up:
    cyp24a1_core_selection_outcome = "included_in_canonical_up_core"
elif in_core_down:
    cyp24a1_core_selection_outcome = "included_in_canonical_down_core"
elif not passes_up_recurrence and not passes_down_recurrence:
    cyp24a1_core_selection_outcome = "fails_minimum_recurrence_threshold"
elif passes_up_recurrence and not in_core_up:
    cyp24a1_core_selection_outcome = "passes_up_recurrence_but_not_final_core"
elif passes_down_recurrence and not in_core_down:
    cyp24a1_core_selection_outcome = "passes_down_recurrence_but_not_final_core"
else:
    cyp24a1_core_selection_outcome = "not_selected"


# -----------------------------------------------------------------------------
# Build compact membership summary table
# -----------------------------------------------------------------------------

cyp24a1_core_membership_check = pd.DataFrame(
    [
        {
            "gene_id": target_gene_id,
            "gene_symbol": target_gene_symbol,
            "canonical_top_n": canonical_top_n,
            "canonical_min_votes": canonical_min_votes,
            "canonical_target_up": canonical_target_up,
            "canonical_target_down": canonical_target_down,
            "up_votes": cyp24a1_up_votes,
            "down_votes": cyp24a1_down_votes,
            "passes_up_recurrence": passes_up_recurrence,
            "passes_down_recurrence": passes_down_recurrence,
            "in_core_up": in_core_up,
            "in_core_down": in_core_down,
            "in_canonical_core": in_canonical_core,
            "core_selection_outcome": cyp24a1_core_selection_outcome,
        }
    ]
)


# -----------------------------------------------------------------------------
# Add per-cell-line detail to the compact summary
# -----------------------------------------------------------------------------

for _, row in cyp24a1_membership_by_cell.iterrows():
    cell_line = row["cell_id"]
    cyp24a1_core_membership_check[f"mean_effect_{cell_line}"] = row["mean_effect"]
    cyp24a1_core_membership_check[f"rank_positive_effect_{cell_line}"] = row["rank_positive_effect"]
    cyp24a1_core_membership_check[f"rank_negative_effect_{cell_line}"] = row["rank_negative_effect"]
    cyp24a1_core_membership_check[f"in_top{canonical_top_n}_up_{cell_line}"] = row[top_up_column]
    cyp24a1_core_membership_check[f"in_top{canonical_top_n}_down_{cell_line}"] = row[top_down_column]


# -----------------------------------------------------------------------------
# Report CYP24A1 core-membership audit
# -----------------------------------------------------------------------------

print("CYP24A1 core-membership audit completed.")
print("")
print("Per-cell-line CYP24A1 core-selection details:")
print(cyp24a1_membership_by_cell.to_string(index=False))
print("")
print("Compact CYP24A1 core-membership summary:")
print(cyp24a1_core_membership_check.to_string(index=False))

## Revision summary: core directionality and CYP24A1 audit

This notebook audited the canonical vitamin D transcriptional core and the behavior of `CYP24A1` in response to reviewer concerns about gene directionality, Figure 2A terminology, and the absence of `CYP24A1` from the reported core.

### Core directionality

The canonical core contained 77 genes: 42 assigned to the UP component and 35 assigned to the DOWN component. Directionality was evaluated across the five analyzed cell lines using the sign of the mean LINCS Level 5 effect within each cell line.

Overall, the core showed strong directional coherence:

* 40 of 77 genes were concordant with the expected direction in all five cell lines.
* 25 of 77 genes were concordant in four of five cell lines.
* 12 of 77 genes were partially concordant.
* No gene was classified as discordant or opposite-dominant.

Thus, most canonical core genes followed the expected UP or DOWN direction in all or nearly all cellular contexts. This supports the interpretation that the core is directionally structured, while also acknowledging that a subset of genes shows context-dependent directionality across cell lines.

### CYP24A1 and Figure 2A

`CYP24A1` was present in the LINCS expression matrix and mapped unambiguously to gene ID `1591`.

Under the Figure 2A ranking criterion, which uses global mean absolute z-score across all curated signatures, `CYP24A1` ranked 11,556 of 12,328 genes, with a mean absolute z-score of 0.440749 and a mean signed z-score of 0.101788.

Therefore, `CYP24A1` is not absent because of missing annotation or omission from the dataset. Rather, it does not appear in Figure 2A because the figure prioritizes genes with the strongest global mean absolute transcriptional response across all signatures, and `CYP24A1` shows relatively low global response magnitude in this curated LINCS subset.

### CYP24A1 and core membership

The canonical core was defined using recurrence across cell-line-level extremes, not by canonical biological expectation. Specifically, genes were selected if they appeared among the top 50 positive or top 50 negative mean effects in at least two cell lines.

`CYP24A1` did not appear among the top 50 positive or top 50 negative genes in any of the five cell lines:

* A549: positive rank 873
* HA1E: positive rank 411
* MCF7: positive rank 3771
* PC3: negative rank 6760
* U2OS: positive rank 5406

Accordingly, `CYP24A1` received 0 UP recurrence votes and 0 DOWN recurrence votes, failing the minimum recurrence threshold required for canonical core membership.

### Manuscript implication

These results support a clarification in the revised manuscript and response letter: `CYP24A1` is present in the analyzed LINCS gene space but does not meet either the Figure 2A global response-magnitude criterion or the recurrence-based canonical core criterion. Its absence should therefore be described as a consequence of the dataset-specific ranking and core-selection definitions, not as evidence against its established role as a canonical vitamin D target gene.
